# NBA Game Prediction - XGBoost Model (GPU Accelerated)

This notebook trains an XGBoost classifier with GPU acceleration to predict NBA game outcomes.


## 1. Import Libraries

**Note:** This notebook uses **XGBoost with GPU acceleration**.  
Make sure to enable GPU in Kaggle notebook settings: Settings → Accelerator → GPU T4 x2


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os
import subprocess

print("Libraries imported successfully!")


## 2. Load Data


In [ ]:
# Load the engineered dataset
# Update the path if your dataset is in a different location
df = pd.read_csv('/kaggle/input/ready-to-train-basketball-data/training_dataset_engineered.csv')

print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")
print(f"Seasons: {df['season'].min()} to {df['season'].max()}")
df.head()


## 3. Prepare Features and Target


## 3b. Reduce Overreliance on `is_home` Feature

**Problem:** Model is too dependent on `is_home` (71.89% importance).  
**Solution:** Create interaction features and optionally downweight `is_home`.


In [ ]:
# Strategy: Create interaction features to make other features more predictive
# This helps the model learn patterns beyond just home/away

print("Creating interaction features to reduce reliance on 'is_home'...")

# DECIDE: Will we remove is_home? If yes, don't create is_home interactions
REMOVE_IS_HOME = True  # Set to False if you want to keep is_home

if REMOVE_IS_HOME and 'is_home' in X.columns:
    print("  ℹ️  Will remove is_home, skipping is_home interaction features")
else:
    # Create interaction features that combine is_home with other important features
    if 'is_home' in X.columns and 'elo_diff' in X.columns:
        X['is_home_x_elo_diff'] = X['is_home'] * X['elo_diff']
        print("  ✓ Created: is_home × elo_diff")

    if 'is_home' in X.columns and 'team1_elo' in X.columns:
        X['is_home_x_team1_elo'] = X['is_home'] * X['team1_elo']
        X['is_home_x_team2_elo'] = X['is_home'] * X['team2_elo']
        print("  ✓ Created: is_home × team ELO interactions")

    if 'is_home' in X.columns and 'plus_minus_diff_l10' in X.columns:
        X['is_home_x_plus_minus'] = X['is_home'] * X['plus_minus_diff_l10']
        print("  ✓ Created: is_home × plus_minus_diff")

# Create team strength interactions (without is_home)
if 'team1_offensive_rating' in X.columns and 'team2_defensive_rating' in X.columns:
    X['off_vs_def_matchup'] = X['team1_offensive_rating'] - X['team2_defensive_rating']
    print("  ✓ Created: offensive vs defensive matchup")

if 'team1_avg_pts_scored_l10' in X.columns and 'team2_avg_pts_allowed_l10' in X.columns:
    X['scoring_vs_defense'] = X['team1_avg_pts_scored_l10'] - X['team2_avg_pts_allowed_l10']
    print("  ✓ Created: scoring vs defense matchup")

# Option 1: Downweight is_home by multiplying by a factor (keeps it but reduces importance)
# Uncomment the line below to reduce is_home impact:
# X['is_home'] = X['is_home'] * 0.5  # Reduce is_home importance by 50%
# print("  ⚠️  Downweighted 'is_home' by 50%")

# Option 2: Remove is_home entirely (forces model to learn from other features)
# This is more aggressive but will force the model to use team strength features
# ACTIVE: Removing is_home to force balanced feature usage
if REMOVE_IS_HOME:
    if 'is_home' in X.columns:
        X = X.drop(columns=['is_home'], inplace=False)
        print("  ⚠️  Removed 'is_home' feature to force model to use other features")
        print("  💡 Model will now rely on team strength, ELO, and form features")
    else:
        print("  ℹ️  'is_home' not found in columns (already removed or never existed)")
    
    # Also remove any is_home interaction features if they exist
    is_home_interactions = [col for col in X.columns if 'is_home' in col.lower()]
    if is_home_interactions:
        X = X.drop(columns=is_home_interactions, inplace=False)
        print(f"  ⚠️  Also removed {len(is_home_interactions)} is_home interaction features")
else:
    print("  ℹ️  Keeping 'is_home' feature (set REMOVE_IS_HOME = True to remove)")

# Update feature_cols list to match X
feature_cols = list(X.columns)

# VERIFY is_home is removed
if 'is_home' in feature_cols:
    print("  ❌ ERROR: is_home still in feature_cols! Removing now...")
    feature_cols.remove('is_home')
    X = X.drop(columns=['is_home'], inplace=False)
    feature_cols = list(X.columns)

if 'is_home' not in X.columns:
    print("  ✅ Verified: 'is_home' successfully removed from features")

print(f"\n✅ Total features after interactions: {len(feature_cols)}")
print(f"   Features: {', '.join(feature_cols[:5])}..." if len(feature_cols) > 5 else f"   Features: {', '.join(feature_cols)}")


In [ ]:
# Columns to exclude (metadata, identifiers, targets)
exclude_cols = [
    'game_id', 'game_date', 'season',
    'team1_id', 'team1_name', 'team2_id', 'team2_name',
    'team1_score', 'team2_score',
    'home_team_won',  # We'll use team1_won instead
]

# Feature columns (all columns except excluded ones)
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Remove target from features if it's in the list
if 'team1_won' in feature_cols:
    feature_cols.remove('team1_won')

print(f"Number of features: {len(feature_cols)}")
print(f"\nFeatures:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

# Prepare X and y
X = df[feature_cols].copy()
y = df['team1_won'].copy()

print(f"\nShape: X={X.shape}, y={y.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")


## 4. Handle Missing Values


In [ ]:
# Check for missing values
missing = X.isnull().sum()
if missing.sum() > 0:
    print("Missing values found:")
    print(missing[missing > 0])
    # Fill missing values with 0 (or median for numeric columns)
    X = X.fillna(0)
    print("\nFilled missing values with 0")
else:
    print("No missing values found!")


## 5. Split Data (Temporal Split)


In [ ]:
# Convert game_date to datetime for sorting
df['game_date'] = pd.to_datetime(df['game_date'])

# Sort by date
sorted_indices = df.sort_values('game_date').index
X_sorted = X.loc[sorted_indices]
y_sorted = y.loc[sorted_indices]

# Temporal split: 80% train, 20% test (older games -> newer games)
split_idx = int(len(X_sorted) * 0.8)

X_train = X_sorted.iloc[:split_idx]
y_train = y_sorted.iloc[:split_idx]
X_test = X_sorted.iloc[split_idx:]
y_test = y_sorted.iloc[split_idx:]

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTrain date range: {df.loc[X_train.index, 'game_date'].min()} to {df.loc[X_train.index, 'game_date'].max()}")
print(f"Test date range: {df.loc[X_test.index, 'game_date'].min()} to {df.loc[X_test.index, 'game_date'].max()}")


## 6. Train XGBoost Model (GPU Accelerated)

**Alternative:** If you prefer Random Forest (CPU only), see the commented section below.


### Optional: Improved Hyperparameters (to reduce overfitting)

If you see significant overfitting (train accuracy >> test accuracy), try these parameters:


In [ ]:
# Uncomment and use these parameters if you see overfitting (train acc >> test acc)
# These parameters add regularization to reduce overfitting:

# model = xgb.XGBClassifier(
#     n_estimators=200,  # More trees but with regularization
#     max_depth=6,  # Lower depth to reduce complexity
#     learning_rate=0.05,  # Lower learning rate for better generalization
#     tree_method='hist',
#     device='cuda' if has_gpu else 'cpu',
#     random_state=42,
#     eval_metric='logloss',
#     subsample=0.8,  # Use 80% of samples per tree
#     colsample_bytree=0.8,  # Use 80% of features per tree
#     reg_alpha=0.1,  # L1 regularization
#     reg_lambda=1.0,  # L2 regularization
#     min_child_weight=3,  # Minimum samples in leaf
#     early_stopping_rounds=10  # Stop if no improvement
# )
# 
# # Convert to arrays first
# X_train_array = X_train.values if hasattr(X_train, 'values') else X_train
# X_test_array = X_test.values if hasattr(X_test, 'values') else X_test
# 
# model.fit(
#     X_train_array, y_train, 
#     eval_set=[(X_test_array, y_test)],
#     verbose=True
# )


In [ ]:
# Check if GPU is available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    has_gpu = result.returncode == 0
    if has_gpu:
        print("✅ GPU detected! XGBoost will use GPU acceleration")
    else:
        print("⚠️  GPU not detected, XGBoost will use CPU")
except:
    has_gpu = False
    print("⚠️  GPU not detected, XGBoost will use CPU")

# IMPROVED PARAMETERS: Reduce overfitting AND reduce is_home reliance
# Key changes:
# - colsample_bytree=0.6: Only use 60% of features per tree (forces diversity!)
# - max_depth=5: Lower depth prevents overfitting and single-feature dominance
# - Higher regularization: Penalizes reliance on single features
# - More trees with lower learning rate: Better generalization

model = xgb.XGBClassifier(
    n_estimators=300,  # More trees for better learning
    max_depth=5,  # Lower depth to reduce complexity and force feature diversity
    learning_rate=0.03,  # Lower learning rate for better generalization
    tree_method='hist',
    device='cuda' if has_gpu else 'cpu',
    random_state=42,
    eval_metric='logloss',
    subsample=0.7,  # Use 70% of samples per tree (more diversity)
    colsample_bytree=0.6,  # Use only 60% of features per tree (forces diversity!)
    colsample_bylevel=0.8,  # Use 80% of features per level
    reg_alpha=0.5,  # Higher L1 regularization (penalizes single-feature reliance)
    reg_lambda=2.0,  # Higher L2 regularization
    min_child_weight=5,  # Higher minimum samples in leaf (prevents overfitting)
    gamma=0.1,  # Minimum loss reduction to split (prevents unnecessary splits)
    max_delta_step=1,  # Constraint on leaf weights
    scale_pos_weight=1,  # Balance for imbalanced classes
    early_stopping_rounds=20  # Stop if no improvement for 20 rounds
)

print("\nTraining XGBoost with improved parameters...")
print("  🎯 Key improvements:")
print("     - colsample_bytree=0.6 (forces feature diversity)")
print("     - max_depth=5 (prevents overfitting)")
print("     - Higher regularization (reduces is_home dominance)")
if has_gpu:
    print("  🚀 Using GPU acceleration for faster training!")
else:
    print("  💻 Using CPU (enable GPU in Kaggle settings for faster training)")

# FINAL CHECK: Ensure is_home is removed before training
if 'is_home' in X_train.columns:
    print("  ❌ WARNING: is_home found in training data! Removing now...")
    X_train = X_train.drop(columns=['is_home'], inplace=False)
    X_test = X_test.drop(columns=['is_home'], inplace=False)
    print("  ✅ Removed is_home from train/test sets")

if 'is_home' in X_train.columns or 'is_home' in X_test.columns:
    print("  ❌ ERROR: Failed to remove is_home! Check your data.")
else:
    print("  ✅ Verified: is_home not in training/test data")

# Convert to arrays and train
X_train_array = X_train.values if hasattr(X_train, 'values') else X_train
X_test_array = X_test.values if hasattr(X_test, 'values') else X_test

model.fit(
    X_train_array, y_train, 
    eval_set=[(X_test_array, y_test)],
    verbose=True
)

print("\n✅ XGBoost training complete!")

# ============================================================================
# ALTERNATIVE: Random Forest (CPU only) - Commented out
# ============================================================================
# Uncomment below if you prefer Random Forest instead of XGBoost
# 
# from sklearn.ensemble import RandomForestClassifier
# 
# model = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=20,
#     min_samples_split=10,
#     min_samples_leaf=5,
#     random_state=42,
#     n_jobs=-1,  # Use all available CPU cores
#     verbose=1
# )
# 
# print("Training Random Forest model...")
# print("⚠️  Note: Random Forest uses CPU only (GPU not used)")
# model.fit(X_train, y_train)
# print("\nTraining complete!")


## 7. Evaluate Model


In [ ]:
# Make predictions
# Convert to numpy arrays to avoid device mismatch warnings
X_train_array = X_train.values if hasattr(X_train, 'values') else X_train
X_test_array = X_test.values if hasattr(X_test, 'values') else X_test

y_train_pred = model.predict(X_train_array)
y_test_pred = model.predict(X_test_array)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)
print(f"\nTraining Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)") 
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Classification report
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, y_test_pred, target_names=['Team2 Won', 'Team1 Won']))                                                                      

# Confusion matrix
print("\n" + "=" * 60)
print("CONFUSION MATRIX (Test Set)")
print("=" * 60)
cm = confusion_matrix(y_test, y_test_pred)
print(cm)
print(f"\nTrue Negatives (Team2 Won predicted as Team2 Won): {cm[0,0]}")        
print(f"False Positives (Team2 Won predicted as Team1 Won): {cm[0,1]}")
print(f"False Negatives (Team1 Won predicted as Team2 Won): {cm[1,0]}")
print(f"True Positives (Team1 Won predicted as Team1 Won): {cm[1,1]}")


## 8. Feature Importance


In [ ]:
# Get feature importances
# Use the actual features the model was trained on (from X_train)
# This ensures we match the model's feature_importances_ array
actual_features = list(X_train.columns)

# Verify lengths match
if len(actual_features) != len(model.feature_importances_):
    print(f"⚠️  Warning: Feature count mismatch!")
    print(f"   X_train columns: {len(actual_features)}")
    print(f"   Model importances: {len(model.feature_importances_)}")
    print(f"   Using model's feature count...")
    # Use model's feature names if available (XGBoost 2.0+)
    if hasattr(model, 'feature_names_in_'):
        actual_features = list(model.feature_names_in_)
    else:
        # Fallback: use first N features where N = len(importances)
        actual_features = actual_features[:len(model.feature_importances_)]

feature_importance = pd.DataFrame({
    'feature': actual_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print("=" * 60)
for idx, row in feature_importance.head(20).iterrows():
    print(f"{row['feature']:35s} {row['importance']:.4f}")

# Visualize (optional)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Model Analysis & Recommendations


In [ ]:
# Model Performance Analysis
print("=" * 70)
print("MODEL ANALYSIS")
print("=" * 70)

# Calculate overfitting gap
overfitting_gap = train_accuracy - test_accuracy
print(f"\n📊 Performance Summary:")
print(f"   Training Accuracy: {train_accuracy:.2%}")
print(f"   Test Accuracy: {test_accuracy:.2%}")
print(f"   Overfitting Gap: {overfitting_gap:.2%}")

if overfitting_gap > 0.15:
    print(f"\n⚠️  WARNING: Significant overfitting detected!")
    print(f"   The model is memorizing training data rather than learning patterns.")
    print(f"   Recommendations:")
    print(f"   - Reduce model complexity (lower max_depth, fewer estimators)")
    print(f"   - Add regularization (higher learning_rate, subsample)")
    print(f"   - Use early stopping")
elif overfitting_gap > 0.10:
    print(f"\n⚠️  Moderate overfitting detected. Consider regularization.")
else:
    print(f"\n✅ Overfitting is within acceptable range.")

# Feature importance insights
print(f"\n📈 Feature Importance Insights:")
print(f"   Most Important: {feature_importance.iloc[0]['feature']} ({feature_importance.iloc[0]['importance']:.1%})")
print(f"   Top 3 Features account for: {feature_importance.head(3)['importance'].sum():.1%} of importance")

if feature_importance.iloc[0]['importance'] > 0.5:
    print(f"\n⚠️  WARNING: Single feature dominates ({feature_importance.iloc[0]['importance']:.1%})")
    print(f"   This suggests the model may be too reliant on one feature.")
    print(f"   Consider:")
    print(f"   - Feature engineering to create more balanced features")
    print(f"   - Regularization to force the model to use more features")

# Baseline comparison
baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())
improvement = test_accuracy - baseline_accuracy
print(f"\n🎯 Baseline Comparison:")
print(f"   Baseline (majority class): {baseline_accuracy:.2%}")
print(f"   Model Improvement: {improvement:.2%} ({improvement*100:.1f} percentage points)")

if improvement < 0.10:
    print(f"\n⚠️  Model improvement is modest. Consider:")
    print(f"   - More feature engineering")
    print(f"   - Hyperparameter tuning")
    print(f"   - Trying different algorithms")
else:
    print(f"\n✅ Model shows meaningful improvement over baseline!")

print("\n" + "=" * 70)


## 10. Save Model


In [ ]:
# Save the model to /kaggle/working (this will be available for download)
model_filename = 'xgboost_nba_model.pkl'
model_path = f'/kaggle/working/{model_filename}'

joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")

# Also save feature names for later use
feature_names_path = '/kaggle/working/feature_names.txt'
with open(feature_names_path, 'w') as f:
    for feature in feature_cols:
        f.write(f"{feature}\n")
print(f"Feature names saved to: {feature_names_path}")

# Verify file exists
if os.path.exists(model_path):
    file_size = os.path.getsize(model_path) / (1024 * 1024)  # Size in MB
    print(f"\nModel file size: {file_size:.2f} MB")
    print("\n✅ Model is ready for download!")
    print("\nTo download:")
    print("1. Go to the 'Output' tab on the right sidebar")
    print("2. Click on 'xgboost_nba_model.pkl'")
    print("3. Click the download button")
else:
    print("\n❌ Error: Model file not found!")


## 11. Quick Prediction Example


In [ ]:
# Example: Make predictions on a few test samples
sample_indices = X_test.index[:5]
sample_X = X_test.loc[sample_indices]
sample_y = y_test.loc[sample_indices]

# Convert to numpy array to avoid device mismatch
sample_X_array = sample_X.values if hasattr(sample_X, 'values') else sample_X

predictions = model.predict(sample_X_array)
probabilities = model.predict_proba(sample_X_array)

print("Sample Predictions:")
print("=" * 80)
for idx, (true_label, pred, prob) in enumerate(zip(sample_y, predictions, probabilities)):
    game_info = df.loc[sample_indices[idx]]
    team1_name = game_info.get('team1_name', 'Team1')
    team2_name = game_info.get('team2_name', 'Team2')
    
    print(f"\nGame {idx+1}:")
    print(f"  {team1_name} vs {team2_name}")
    print(f"  Actual: {'Team1 Won' if true_label == 1 else 'Team2 Won'}")
    print(f"  Predicted: {'Team1 Won' if pred == 1 else 'Team2 Won'}")
    print(f"  Confidence: {prob[max(pred, 0)]*100:.1f}%")
    print(f"  {'✅ Correct' if true_label == pred else '❌ Incorrect'}")
